In [ ]:
import re

In [ ]:
with open("daten/fraudulent_emails.txt", "r", encoding="ISO-8859-1") as emails:
    amounts = []
    for line in emails:
        # Wir suchen nur Beträge, die mit US$ anfangen
        m = re.search(r'USD? *\$? *(\d[0-9., ]+)([M]?)', line, re.IGNORECASE)
        if not m:
            continue
        # Leerzeichen löschen
        number = re.sub(' ', '', m.group(1))

        parts = re.split(r'[.,]', number)
        # Manche Zahlen haben einen Punkt am Ende. Den ignorieren wir
        if len(parts[-1]) == 0:
            parts = parts[:-1]

        # Ist letzter Block der mit Nachkommastellen?
        # Ja, wenn min. 2 Blöcke und entweder weniger als 3 Stellen oder ein M folgt
        if len(parts) > 1 and (len(parts[-1]) < 3 or m.group(2) in ['m', 'M']):
            amount = float(''.join(parts[:-1]) + '.' + parts[-1])
        else:
            amount = float(''.join(parts))

        # M meint "Millionen USD", aber nur wenn Zahl <1000
        if m.group(2) in ['m', 'M'] and amount < 1000:
            amount *= 1000000
        
        # zusätzliche Korrekturen
        if amount < 100:
            amount *= 1000000
        if amount < 10000:
            amount *= 1000
        # Wir fügen nur Werte kleiner 1Mrd und größer 1Mio hinzu:
        if amount > 1e6 and amount <= 1e9:
            amounts.append(amount)            
        else:
            print(amount, m.group(1), m.group(2))

In [ ]:
import matplotlib.pyplot as plt

# Ungefiltert
plt.hist(amounts, bins=30)
plt.xlabel('USD')
plt.ylabel('Anzahl')
plt.title('Versprochener Geldsegen')
plt.show()

# Zoom auf Beträge <100M USD
amountsM = [amount/1e6 for amount in amounts if amount < 100e6]
plt.hist(amountsM, bins=30)
plt.xlabel('Millionen USD')
plt.ylabel('Anzahl')
plt.title('Versprochener Geldsegen')
plt.show()